
The package accelerate (from Intel) is not explicitly used in the codes below.But it's required for using the Trainer class from the transformers library with PyTorch. However, the actual code didn't use accelerate directly.

The package datasets is a Python library provided by Hugging Face (https://huggingface.co/). It's a widely-used toolkit for working with natural language processing (NLP) datasets.

In [1]:
!pip install datasets torch
!pip install accelerate -U
!pip install transformers -U



  Using cached pyarrow-23.0.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.1 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 13.9 MB/s  0:00:00
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached pyarrow-23.0.1-cp312-cp312-macosx_12_0_arm64.whl (34.2 MB)
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/13 [pyarrow]
    Found existing installation: fsspec 2026.3.0━━━━━━━━━━━━━━  1/13 [pyarrow]
    Uninstalling fsspec-2026.3.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/13 [pyarrow]
      Successfully uninstalled fsspec-2026.3.0━━━━━━━━━━━━━━━━  1/13 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [datasets]/13 [datasets

In [ ]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
# Decode text output
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

new_text = 'to be or not to be'

# 1. this is the same as tokenizer.tokenize(text2)
# 2. the to(device) at the end is very important -- otherwise this quantity exists on cpu
input_ids = tokenizer.encode(new_text, return_tensors="pt").to(device)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
input_ids

tensor([[1462,  307,  393,  407,  284,  307]], device='cuda:0')

In [ ]:
# define model
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# Generate output
output = model.generate(input_ids, max_length=50)

print(output[0])

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


tensor([1462,  307,  393,  407,  284,  307,   13,  198,  198,  464,  717, 1517,
         284,  466,  318,  284,  787, 1654,  326,  345,  389,  407, 1262,  262,
        2642, 3788,   13,  198,  198, 1532,  345,  389, 1262,  257, 1180, 3788,
          11,  345,  815,  307, 1498,  284,  779,  340, 1231,  597, 2761,   13,
         198,  198], device='cuda:0')


model.safetensors: the SafeTensor format, a file format used by the Transformers library (which includes GPT2) for storing and loading model weights. 548M: This shows the total size of the model (548MB).

generation_config.json: a configuration file used by the Transformers library (which includes GPT2) to store settings and parameters for text generation. This file contains options and hyperparameters that control the behavior of the model when generating text, such as:

- Maximum length of the generated text
- Number of samples to generate
- Temperature for sampling (affects the randomness of the output)
- Top-k filtering (controls the diversity of the output)

About top-k filtering: The model generates a list of possible next tokens, ranked by their probability (how likely they are to appear next).
The top-k filter selects only the top k tokens with the highest probabilities.
The model then randomly selects one of these top k tokens as the next output token.

In [ ]:
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

to be or not to be.

The first thing to do is to make sure that you are not using the wrong software.

If you are using a different software, you should be able to use it without any problems.




In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
from transformers import GPT2LMHeadModel

# Load the pre-trained GPT-2 model
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Resize the token embeddings in case you added special tokens
model.resize_token_embeddings(len(tokenizer))


Embedding(50257, 768)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',          # Directory for saving trained model
    num_train_epochs=3,              # Number of training epochs
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # Batch size for training
    per_device_eval_batch_size=8,    # Batch size for evaluation
    warmup_steps=500,                # Number of warmup steps
    weight_decay=0.01,               # Weight decay to prevent overfitting
    logging_dir='./logs',            # Directory for storing logs
    evaluation_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",           # Save the model at the end of each epoch
    load_best_model_at_end=True      # Load the best model at the end of training
)


In [ ]:
'''
Each of train, validation (here called 'unsupervised'), and test sets contains 25,000 examples

In this example, val/unsupervised and test tests are interchangeable.

If you use one of them to evaluate the model and tune hyperparameters, then leave the other one
as the final test set.


 '''


from transformers import GPT2Tokenizer
from datasets import load_dataset

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Set the padding token to EOS token if it's not already set
# This is important!!
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset
dataset = load_dataset("imdb")

# Define the tokenize function
def tokenize_function(examples):
    # Tokenize the inputs and labels
    return tokenizer(examples['text'], max_length=512, truncation=True, padding="max_length")

# Apply the tokenize function to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=dataset['train'].column_names)


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
tokenized_datasets.keys()

dict_keys(['train', 'test', 'unsupervised'])

In [ ]:
'''

Masked Language Model (mlm) vs. Causal Language Model

In MLM, during training, some percentage of the input tokens are randomly masked
(i.e., replaced with a special token like [MASK]), and the objective of the model
is to predict the original value of these masked tokens based only on their context.
BERT is an example.


Causal language modeling uses the preceding tokens to predict the next token.
GPT is an example.

'''


from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Important: GPT-2 should not use masked language modeling
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator,
)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
model.save_pretrained('/content/drive/MyDrive/colab-models/finetuned_gpt2_imdb')

In [ ]:
model2 = GPT2LMHeadModel.from_pretrained('/content/drive/MyDrive/colab-models/finetuned_gpt2_imdb').to(device)


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


[{'generated_text': 'As a movie critic, I believe that my critical-review of this movie should be a guide and for me, so I feel that the movie deserves more than I have received. On many occasions when I have been reading the commentaries that are coming'}]


As a movie critic, I believe anything that makes a new movie in the film business is a good movie. And, I am glad that director/writer/producer/actor Bill Pullman, when it came out in the U.S,


In [ ]:
import torch

# Assuming you are using CUDA if it's available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load your model to the device
model2.to(device)

# Prepare your input text
prompt = "It's funny that"

# Encode the input text and send to the same device
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

# Generate text
output = model2.generate(input_ids, max_length=100, num_return_sequences=1, no_repeat_ngram_size=2, early_stopping=True)

# Decode the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


It's funny that I'm not a big fan of the "The Simpsons" series, but I do like the show. I've seen it a few times, and I still love it. It's a great show, with a lot of humor, a good cast, great characters, good writing, lots of action, some great humor and a very good story. The show is very funny, it's very well written, the characters are very likable, they're all likables, there


In [ ]:
'''
- ngram

   In the sentence "The cat sat on the mat," the bigrams would be
   ["The cat", "cat sat", "sat on", "on the", "the mat"].

   The trigrams would be ["The cat sat", "cat sat on", "sat on the", "on the mat]

- no_repeat_ngram_size=2

prevents the repetition of 2-gram sequences within the generated text.


- early_stopping

To halt the text generation process before reaching the maximum specified length
if the model outputs the end-of-sequence token (EOS).

'''

prompt = "To be or not to be,"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

# Generate text
output = model.generate(
    input_ids,
    max_length=100,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    early_stopping=True
)

# Decode the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset

# Load tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Load the Tiny Shakespeare dataset
dataset = load_dataset("tiny_shakespeare")



/usr/local/lib/python3.10/dist-packages/datasets/load.py:1486: FutureWarning: The repository for tiny_shakespeare contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/tiny_shakespeare
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
print(len(dataset['train'][0]['text']))

1003854


In [ ]:
print(dataset['train'][0]['text'][0:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
